# 03 — Model Training and Results

Train the GRU model on MRMR-selected channels and visualize results.

This notebook runs the full training pipeline and displays:
1. Training curves
2. Prediction vs actual force
3. Scatter plot with R² metric
4. Per-subject performance breakdown

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt

from src.config import RESULTS_DIR, MODELS_DIR, FIGURES_DIR

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Run Training

Execute the training pipeline. This will:
- Load MRMR results (run notebook 02 first, or use `--all` flag)
- Preprocess data (RMS envelope, sequencing)
- Train GRU with early stopping
- Evaluate and save results

**Note**: Training takes several minutes depending on hardware.

In [ ]:
# Option 1: Run training from this notebook
# Uncomment and run if you haven't trained yet:

# !python -m src.models.train --dataset ninapro

# Option 2: If already trained, just load results
print("Loading saved results...")
results_path = RESULTS_DIR / 'ninapro_results.json'

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print(f"Dataset: {results['dataset']}")
    print(f"Selected channels: {results['selected_channel_names']}")
    print(f"Training samples: {results['n_train']:,}")
    print(f"Test samples: {results['n_test']:,}")
else:
    print(f"No results found at {results_path}")
    print("Run training first: python -m src.models.train --dataset ninapro")

## 2. Overall Metrics

In [ ]:
if results_path.exists():
    metrics = results['metrics']
    
    print("=" * 50)
    print(f"  Model Performance — {results['dataset'].upper()}")
    print("=" * 50)
    print(f"  R²:          {metrics['R2']:.4f}")
    print(f"  RMSE:        {metrics['RMSE']:.4f}")
    print(f"  NRMSE:       {metrics['NRMSE_pct']:.2f}%")
    print(f"  MAE:         {metrics['MAE']:.4f}")
    print(f"  Correlation: {metrics['correlation']:.4f}")
    print("=" * 50)
    print(f"  Training epochs: {results['training']['epochs_run']}")
    print(f"  Training time:   {results['training']['train_time_s']}s")
    print(f"  Total params:    {results['training']['total_params']:,}")
else:
    print("No results available. Run training first.")

## 3. Training Curves

In [ ]:
# Display saved training curves plot
from IPython.display import Image, display
from pathlib import Path

curves_path = FIGURES_DIR / 'training_curves_ninapro.png'
if curves_path.exists():
    display(Image(filename=str(curves_path), width=700))
else:
    print(f"Plot not found at {curves_path}. Run training first.")

## 4. Prediction vs Actual Force

In [ ]:
pred_path = FIGURES_DIR / 'prediction_vs_actual_ninapro.png'
if pred_path.exists():
    display(Image(filename=str(pred_path), width=800))
else:
    print(f"Plot not found at {pred_path}. Run training first.")

## 5. Scatter Plot

In [ ]:
scatter_path = FIGURES_DIR / 'scatter_ninapro.png'
if scatter_path.exists():
    display(Image(filename=str(scatter_path), width=600))
else:
    print(f"Plot not found at {scatter_path}. Run training first.")

## 6. Per-Subject R² Breakdown

In [ ]:
if results_path.exists():
    per_subj = results['per_subject']
    r2_values = [s['R2'] for s in per_subj]
    sids = [s['subject_id'] for s in per_subj]
    
    print(f"Per-Subject R² Statistics:")
    print(f"  Mean:   {np.mean(r2_values):.4f}")
    print(f"  Median: {np.median(r2_values):.4f}")
    print(f"  Std:    {np.std(r2_values):.4f}")
    print(f"  Min:    {np.min(r2_values):.4f} (Subject {sids[np.argmin(r2_values)]})")
    print(f"  Max:    {np.max(r2_values):.4f} (Subject {sids[np.argmax(r2_values)]})")
    print(f"  R² > 0.7: {sum(1 for r in r2_values if r > 0.7)}/{len(r2_values)} subjects")
    print(f"  R² > 0.5: {sum(1 for r in r2_values if r > 0.5)}/{len(r2_values)} subjects")

# Display saved plot
r2_path = FIGURES_DIR / 'per_subject_r2_ninapro.png'
if r2_path.exists():
    display(Image(filename=str(r2_path), width=800))
else:
    print(f"Plot not found at {r2_path}.")

## 7. MRMR Selection Visualization

In [ ]:
mrmr_path = FIGURES_DIR / 'mrmr_ranking_ninapro.png'
if mrmr_path.exists():
    display(Image(filename=str(mrmr_path), width=800))

redundancy_path = FIGURES_DIR / 'redundancy_heatmap_ninapro.png'
if redundancy_path.exists():
    display(Image(filename=str(redundancy_path), width=700))

## 8. Export for Arduino

Export the trained model as a C header file for Arduino deployment.

In [ ]:
# Uncomment to export:
# !python -m src.models.export_model --dataset ninapro

from pathlib import Path
weights_path = Path('..') / 'hardware' / 'arduino' / 'model_weights.h'
if weights_path.exists():
    size_kb = weights_path.stat().st_size / 1024
    print(f"model_weights.h exists ({size_kb:.1f} KB)")
else:
    print("model_weights.h not yet generated.")
    print("Run: python -m src.models.export_model --dataset ninapro")

## 9. Summary

| Component | Detail |
|-----------|--------|
| Channel selection | MRMR (2 channels from 10) |
| Model | GRU(50) -> Dense(100) -> Dense(1) |
| Input | 50 timesteps x 2 channels (1.25s window) |
| Output | Continuous grip force |
| Training | MSE loss, Adam, early stopping |
| Deployment | Arduino Mega 2560 + InMoov hand |